## 1. Last inn data

In [1]:
import pandas as pd

df = pd.read_excel("../data/PCOS_data_without_infertility.xlsx", sheet_name="Full_new")
df.columns = df.columns.str.strip()
df = df.drop(columns=["Sl. No", "Patient File No.", "Unnamed: 44"], errors="ignore")

## 2. Rensing

In [2]:
df = df.rename(columns={"Cycle length(days)": "Bleeding days"})

In [3]:
df = df[df["Cycle(R/I)"].isin([2, 4])].copy()
df["Irregular cycle"] = (df["Cycle(R/I)"] == 4).astype(int)
df["Fast food (Y/N)"] = df["Fast food (Y/N)"].fillna(df["Fast food (Y/N)"].mode()[0])

## 3. Sammenligning av feature-sett

**Funn:** Fast food og Waist:Hip Ratio gir ingen merkbar forbedring. Alle varianter har recall 0,81 og AUC 0,88–0,89. Vi fjerner begge: fast food fordi den er subjektiv og kan ha omvendt årsakssammenheng, og midje/hofte fordi det krever målebånd uten å gi bedre resultater. Videre bruker vi feature-settet «Uten fast food».

In [4]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

y = df["PCOS (Y/N)"]
grunn = ["Age (yrs)", "BMI", "Irregular cycle", "Bleeding days",
         "Weight gain(Y/N)", "hair growth(Y/N)", "Skin darkening (Y/N)",
         "Hair loss(Y/N)", "Pimples(Y/N)", "Reg.Exercise(Y/N)"]

varianter = {
    "Uten fast food": grunn,
    "Med fast food": grunn + ["Fast food (Y/N)"],
    "Uten fast food + midje/hofte": grunn + ["Waist:Hip Ratio"],
    "Med fast food + midje/hofte": grunn + ["Fast food (Y/N)", "Waist:Hip Ratio"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
modell = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))

for navn, features in varianter.items():
    res = cross_validate(modell, df[features], y, cv=cv, scoring=["recall", "precision", "roc_auc"])
    print(f"{navn:32} recall={res['test_recall'].mean():.2f}  "
          f"precision={res['test_precision'].mean():.2f}  AUC={res['test_roc_auc'].mean():.2f}")

Uten fast food                   recall=0.81  precision=0.70  AUC=0.88
Med fast food                    recall=0.81  precision=0.71  AUC=0.89
Uten fast food + midje/hofte     recall=0.81  precision=0.69  AUC=0.88
Med fast food + midje/hofte      recall=0.81  precision=0.70  AUC=0.88


## 4. Logistisk regresjon vs. Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier

features = grunn

modeller = {
    "Logistisk regresjon": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
}

for navn, m in modeller.items():
    res = cross_validate(m, df[features], y, cv=cv, scoring=["recall", "precision", "roc_auc"])
    print(f"{navn:22} recall={res['test_recall'].mean():.2f} (±{res['test_recall'].std():.2f})  "
          f"precision={res['test_precision'].mean():.2f}  AUC={res['test_roc_auc'].mean():.2f}")

Logistisk regresjon    recall=0.81 (±0.05)  precision=0.70  AUC=0.88
Random Forest          recall=0.79 (±0.05)  precision=0.74  AUC=0.88


**Funn:** Logistisk regresjon og Random Forest presterer nesten likt (AUC 0,88 for begge). Forskjellene i recall og precision er mindre enn variasjonen mellom foldene (±0,05). Vi velger logistisk regresjon fordi den har litt høyere recall, er enklere å tolke, og gir sannsynligheter som passer godt til trafikklyset.

In [6]:
## 5. Testsett og terskler
from sklearn.model_selection import train_test_split, cross_val_predict
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    df[features], y, test_size=0.2, stratify=y, random_state=42
)

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))

# Sannsynligheter fra kryssvalidering på treningsdataene
prob = cross_val_predict(lr, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (prob >= t).astype(int)
    tp = ((pred == 1) & (y_train == 1)).sum()
    fp = ((pred == 1) & (y_train == 0)).sum()
    recall = tp / (y_train == 1).sum()
    precision = tp / max(tp + fp, 1)
    print(f"terskel {t:.1f}: recall={recall:.2f}  precision={precision:.2f}")

terskel 0.2: recall=0.89  precision=0.49
terskel 0.3: recall=0.86  precision=0.58
terskel 0.4: recall=0.83  precision=0.63
terskel 0.5: recall=0.82  precision=0.68
terskel 0.6: recall=0.75  precision=0.74
terskel 0.7: recall=0.67  precision=0.77


**Funn og valg av terskler:** Med terskel 0,2 fanger modellen 89 % av PCOS-tilfellene. Ved 0,6 møtes recall og precision (0,75/0,74). Vi bruker derfor grønt under 0,2, gult mellom 0,2 og 0,6, og rødt fra 0,6. Sannsynlighetene er påvirket av klassevekting og vises ikke som prosent på nettsiden.

In [7]:
## 6. Endelig modell
import joblib
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

lr.fit(X_train, y_train)
prob_test = lr.predict_proba(X_test)[:, 1]

GRONN, ROD = 0.2, 0.6

def trafikklys(p):
    return "grønt" if p < GRONN else ("rødt" if p >= ROD else "gult")

print("AUC på testsett:", round(roc_auc_score(y_test, prob_test), 2))
print(confusion_matrix(y_test, (prob_test >= ROD).astype(int)))
print(classification_report(y_test, (prob_test >= ROD).astype(int)))
print(pd.crosstab(pd.Series([trafikklys(p) for p in prob_test], name="Lys"),
                  y_test.reset_index(drop=True).rename("PCOS")))
import os
os.makedirs("../model", exist_ok=True)

joblib.dump({"model": lr, "features": features, "terskler": (GRONN, ROD)}, "../model/pcos_model.joblib")

AUC på testsett: 0.91
[[65  8]
 [ 8 27]]
              precision    recall  f1-score   support

           0       0.89      0.89      0.89        73
           1       0.77      0.77      0.77        35

    accuracy                           0.85       108
   macro avg       0.83      0.83      0.83       108
weighted avg       0.85      0.85      0.85       108

PCOS    0   1
Lys          
grønt  37   2
gult   28   6
rødt    8  27


['../model/pcos_model.joblib']